In [ ]:
import requests
import urllib.request
from bs4 import BeautifulSoup

def getMoviesImg():
    url = requests.get('http://news.jstv.com/a/20171018/1508322421993.shtml')
    #获取网站数据
    html = url.text
    #解析网页
    soup = BeautifulSoup(html,'html.parser')
    #获取所有的img标签
    movie = soup.find_all('img')
    x = 1
    for i in movie:
        # 获取src路径
        imgsrc = i.get('src')
        #判断图片src路径是否以指定内容开头（过滤页面中的其它不想要的图片）
        if imgsrc.startswith('http://static.jstv.com/gather/hl/20171018/'):
            # print(imgsrc)
            #本地路径
            filename = 'E:/download/movie/%s.jpg'%x
            #将URL表示的网络对象复制到本地文件
            urllib.request.urlretrieve(imgsrc , filename)
            print('下载第%d张' % x)
            x += 1
    print('**下载完成!**')

getMoviesImg()


In [ ]:
import requests

import urllib.request

from lxml import etree


url='https://www.tujidao01.com/a/?id=53250'

headers = {
    'User-Agent': 'Mozilla/5.0 (Linux; Android 6.0; Nexus 5 Build/MRA58N) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/68.0.3423.2 Mobile Safari/537.36'

}

response = requests.get(url,headers)
content =response.content.decode('utf-8')
print(content)

HTML = requests.get(url,headers=headers).content
xp_html = etree.HTML(html)


imgnames = xp_html.xpath('//div/p/img/@alt')

imgurls = xp_html.xpath('//div/p/img/@src')

for (imgname,imgurl) in zip(imgnames,imgurls):

    try:

        urllib.request.urlretrieve(imgurl,'C:\PDF_File\%s.jpg' % imgname)


    except Exception as e:

        print(imgname + '：下载出错，地址为：'+ imgurl)

        print('下载完成')

In [ ]:
import requests  ##安装后导入第三方模块 requests（HTTP 客户端库）
import parsel    ##安装后导入第三方模块 parsel（数据解析）
import os        ##系统自带模块，无需安装，直接导入第三方模块 os（操作系统交互功能）

for page in range(1, 6):#构建翻页的范围，从1开始到6（即第5页）结束
    print('=======================正在爬取第{}页数据====================='.format(page))
    # 1、确定爬取的url路径，headers参数
    base_url = 'https://www.tujidao01.com/u/?action=gengxin'.format(page) ##构建一个base_url来存放URL地址
    ##构建一个“.format(page)”来传入页数
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/85.0.4181.9 Safari/537.36'}
    #构建一个hearders参数来伪装为一个浏览器用户，构造出一个身份

    # 2、发送请求 -- requests 模拟浏览器发送请求，获取响应数据
    response = requests.get(base_url, headers=headers)
    #调出静态网页的get方法，获取该网页的URL，将关键字base_url和headers传入进方法中去，并创建一个response对象来接收
    data = response.text
    #从response对象中获取数据，因为数据是字符串类型的所以用".text"来提取，并建立一个data变量来接收
    # print(data)##确定代码是否可以成功运行，如果成功运行则可以注释掉

    # 3、解析数据 -- parsel  转化为Selector对象，Selector对象具有xpath的方法，能够对转化的数据进行处理
    html_data = parsel.Selector(data)
    #转换对象，将data数据传递进变量html_data中，即将data数据自动转换为Selector对象
    data_list = html_data.xpath('//div[@class="Left_bar"]//ul/li/a/@href|//div[@class="Left_bar"]//ul/li/a/img/@title').extract()
    #获取相册的名字，返回的是一个列表
    #将转化为Selector对象的data_list运用xpath,在div中跨节点找到“class="Left_bar"进行精确定位
    # 再按照同样跨节点的方式依次找到<ul>，<li> ,<a>,@a标签中的href属性，再用“.extract()”方法将Selector数据取出，并创建一个data_list变量来接收
    #  print(data_list)

    # 使用列表推导式对列表进行分组
    data_list = [data_list[i:i + 2] for i in range(0, len(data_list), 2)]#将相册的名称和相册的url地址进行分组
    # print(data_list)

    # 遍历列表元素
    for alist in data_list:
        html_url = alist[0]#取到每一个相册的URl地址
        file_name = alist[1]#取到每一个相册的名称
        # print(html_url, file_name)

        # 创建图片的文件夹
        root = 'G:/COS1/'
        if not os.path.exists('img\\' + file_name):
            os.mkdir('img\\' + file_name)#如果没有存在当前文件夹，则创建文件夹
        print('正在下载：', file_name)#打印出正在下载的图片名称

        # 发送详情页的请求,解析出总页数
        response_2 = requests.get(html_url, headers=headers).text
        html_2 = parsel.Selector(response_2)
        page_num = html_2.xpath('//div[@class="ptitle"]//em/text()').extract_first()
        # print(page_num)

        for url in range(1, int(page_num) + 1):
            # 构建相册翻页的url地址
            url_list = html_url.split('.')
            all_url = url_list[0] + '.' + url_list[1] + '.' + url_list[2] + '_' + str(url) + '.' + url_list[3]
            #嵌套出当前相册的每一张图片的URL地址，并拼接
            # print(all_url)

            # 发送详情页的请求,解析详情页的图片url地址
            response_3 = requests.get(all_url, headers=headers).text
            html_3 = parsel.Selector(response_3)
            img_url = html_3.xpath('//div[@class="pic-meinv"]//img/@data-original').extract_first()
            #因为仅当他加载图片时才返回图片数据，所以这个网页是软加载图片
            # 将转化为Selector对象的html_3运用xpath,在div中跨节点找到“class="pic-meinv"进行精确定位
            # 再按照同样跨节点的方式依次找到<img>，@a标签中的hdata-original属性，并创建一个img_url变量来接收
            #使用“.extract_first()”提取出整一个数据，如果不加则只有一张图片
            # print(img_url)

            # 请求图片的url地址
            img_data = requests.get(img_url, headers=headers).content

            # 图片的文件名
            img_name = str(url) + '.jpg'#准备文件名称
            #取当前for循环的索引做为文件名

            # 4、保存数据
            with open('img\\{}\\'.format(file_name) + img_name, 'wb') as f:
                print('下载完成：', img_name)
                f.write(img_data)#写入文件数据


In [ ]:
import requests
from bs4 import BeautifulSoup
import os
import re

def getHtmlurl(url):         #获取网址
    try:
       r=requests.get(url)
       r.raise_for_status()
       r.encoding=r.apparent_encoding
       return r.text
    except:
        return ""

def getimgurl(img):
    href=img['href']
    url="http://www.netbian.com"+href
    #print(url)
    htm=getHtmlurl(url)
    soup=BeautifulSoup(htm,'html.parser')
    return soup

def getpic(html): #获取图片地址并下载
    soup =BeautifulSoup(html,'html.parser')
    all_img=soup.find('div',class_='list').find('ul').find_all("a",attrs={'href':re.compile('^((?!http).)*$'),'target':'_blank'})

    for img in all_img:
       title=img['title']
       if title.find(u"女") != -1 :
        print("不符合要求，跳过")
        continue

       soup1=getimgurl(img)
       im1=soup1.find('div',id='main').find('div',class_='endpage').find('div',class_='pic').find('p').find('a')
       soup2=getimgurl(im1)
       im2=soup2.find('div',id='main').find('table').find('a')

       img_url = im2['href']
       print (img_url)

       #root='/home/suwex/图片/'
       root='/home/suwex/test2/'
       title=title.replace(',','|')
       title=title.replace(' ','|')
       path = root + title

       try:                              #创建或判断路径图片是否存在并下载
           if not os.path.exists(root):
               os.mkdir(root)
           if not os.path.exists(path):
               r = requests.get(img_url)
               with open(path, 'wb') as f:
                   f.write(r.content)
                   f.close()
                   print("文件保存成功")
           else:
               print("文件已存在")
       except:
           print("爬取失败")



def main():
    for i in range(1,10):
      if i==1:
        url='http://www.netbian.com/weimei/index.htm'
      else:
        url='http://www.netbian.com/weimei/index_' + str(i) +'.htm'
      html=(getHtmlurl(url))
      print(str(i)+" : ")
      print(getpic(html))
main()

In [ ]:
import requests
import json
import urllib

def getSogouImag(category,length,path):

    n = length

    cate = category

    imgs = requests.get('http://pic.sogou.com/pics/channel/getAllRecomPicByTag.jsp?category='+cate+'&tag=%E5%85%A8%E9%83%A8&start=0&len='+str(n))

    jd = json.loads(imgs.text)

    jd = jd['all_items']

    imgs_url = []

    for j in jd:

        imgs_url.append(j['bthumbUrl'])

    m = 0

    for img_url in imgs_url:

        print('***** '+str(m)+'.jpg *****'+'  Downloading...')

        urllib.request.urlretrieve(img_url,path+str(m)+'.jpg')

        m = m + 1
    print('Download complete!')

getSogouImag('壁纸',2000,'C:\PDF_File\img')

In [29]:
import requests
import os

url = "https://timgsa.baidu.com/timg?image&quality=80&size\
=b9999_10000&sec=1590932049656&di=bf69222ddaff5a2544610717d22138cc&\
imgtype=0&src=http%3A%2F%2F01.minipic.eastday.com%2F20170519%2F20170\
519153002_902da84e08dac8d68f189cb6c8ea4626_10.jpeg" #图片地址
root = "C://PDF_File//img" #存放图片路径
path = root + '帕吉.jpg'  #设置图片名称及其格式
try: #try···except结构，返回链接错误类型
    headers = {'user-agent':'Mozilla/5.0'} #请求头模拟
    #使用库os的方法确认文件路径是否存在，若不存在则创建
    if not os.path.exists(root):  
        os.mkdir(root)            #若不存在则创建
    if not os.path.exists(path):
        r = requests.get(url, headers=headers)
        with open (path, 'wb') as f:#"wb'表示对二进制文件的写入
            f.write(r.content)  #r.content表示返回内容的二进制形式
            f.close()#with··· as 语句会自动关闭句柄，可不写close()
            print("文件保存成功")
            
    else:
        print('文件已经存在')
        
except ConnectionError as err:
        print(err)



文件保存成功


In [30]:
import requests
from time import time
from threading import Thread

# 继承Thread类创建自定义的线程类
class DownloadHanlder(Thread):

    def __init__(self, url):
        super().__init__()
        self.url = url

    def run(self):
        filename = self.url[self.url.rfind('/') + 1:]  
        #print(filename)
        resp = requests.get(self.url)
        with open('E:/sublime_python/pics' + filename, 'wb') as f:
            f.write(resp.content)


def main():
    resp = requests.get('http://api.tianapi.com/meinv/?key=82bc97e2f2e2e560dd8ddcce55b93c0b&num=10')
    # 将服务器返回的JSON格式的数据解析为字典（Requests中有一个内置的 JSON 解码器——json()方法，助你处理 JSON 数据）
    data_model = resp.json()
    for mm_dict in data_model['newslist']:			# ？？？？
        url = mm_dict['picUrl']
        # 通过多线程的方式实现图片下载
        DownloadHanlder(url).start()
    print()
    print('完成')


if __name__ == '__main__':
    main()


KeyError: 'newslist'